# Shape Network Training Pipeline

Train Shape Network để predict MANO parameters (beta, theta, global_orient, translation) từ ảnh input.

**Components:**
1. Shape Network (ResNet50 backbone) → predict 61 MANO params
2. MANO Layer (PyTorch smplx) → convert params → 3D joints/vertices
3. Loss: 3D joint loss + 2D reprojection + parameter regularization

**Yêu cầu:**
- Python environment với TensorFlow + PyTorch + smplx
- MANO model file (.pkl)

## 1. Check Environment and Imports

In [ ]:
import sys
from pathlib import Path

# Setup path
REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

# Check TensorFlow
try:
    import tensorflow as tf
    print(f"TensorFlow: {tf.__version__}")
    print(f"CUDA available: {tf.test.is_built_with_cuda()}")
except ImportError as e:
    print(f"TensorFlow not available: {e}")

# Check PyTorch (for MANO)
try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"PyTorch not available: {e}")

# Check smplx
try:
    from smplx import MANO
    print("smplx: OK")
except ImportError as e:
    print(f"smplx not available: {e}")

## 2. Test Shape Network + MANO Pipeline

In [ ]:
from __future__ import annotations

import numpy as np
import tensorflow as tf

from models.shape_net import create_shape_net, split_mano_params, TOTAL_PARAMS
from models.mano_layer import get_mano_layer

print("=" * 60)
print("Testing Shape Network + MANO Pipeline")
print("=" * 60)

# Create model
print("\n[1] Creating Shape Network...")
model = create_shape_net(backbone_name="resnet50", trainable_backbone=True)
model.summary()

trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
print(f"Trainable params: {trainable_params:,}")

# Test forward pass
print("\n[2] Testing forward pass...")
batch_size = 2
test_input = tf.random.normal((batch_size, 224, 224, 3))
pred_params = model(test_input, training=False)

print(f"  Input: {test_input.shape}")
print(f"  Output params: {pred_params.shape}")

# Split parameters
beta, theta, global_orient, trans = split_mano_params(pred_params)
print(f"  beta: {beta.shape} (expected: (2, 10))")
print(f"  theta: {theta.shape} (expected: (2, 45))")
print(f"  global_orient: {global_orient.shape} (expected: (2, 3))")
print(f"  trans: {trans.shape} (expected: (2, 3))")

# Test MANO layer
print("\n[3] Testing MANO layer...")
try:
    mano_layer = get_mano_layer()
    vertices, joints = mano_layer(beta, theta, global_orient, trans)
    print(f"  vertices: {vertices.shape} (expected: (2, 778, 3))")
    print(f"  joints: {joints.shape} (expected: (2, 21, 3))")
    print("✓ MANO layer test PASSED")
except Exception as e:
    print(f"✗ MANO layer test FAILED: {e}")
    print("  Note: This requires MANO model file (.pkl)")

# Test gradient flow
print("\n[4] Testing gradient flow...")
with tf.GradientTape() as tape:
    pred_params = model(test_input, training=True)
    beta, theta, global_orient, trans = split_mano_params(pred_params)
    
    # Simple loss: regularization only
    loss = tf.reduce_mean(tf.square(beta)) + tf.reduce_mean(tf.square(theta))

grads = tape.gradient(loss, model.trainable_variables)

has_grads = any(g is not None for g in grads)
print(f"  Gradients computed: {has_grads}")

if has_grads:
    grad_norm = tf.add_n([tf.reduce_sum(tf.square(g)) for g in grads if g is not None])
    print(f"  Gradient norm: {float(grad_norm):.4f}")

print("\n" + "=" * 60)
print("Pipeline test COMPLETE")
print("=" * 60)

## 3. Configuration

In [ ]:
# ========== TRAINING CONFIGURATION ==========

# Dataset
BATCH_SIZE = 16               # Adjust based on VRAM
IMAGE_SIZE = 224

# Model
BACKBONE_MODEL = "resnet50"   # or "mobilenetv3small"
TRAINABLE_BACKBONE = True

# Training
INITIAL_LEARNING_RATE = 1e-4
DECAY_STEPS = 1000
DECAY_RATE = 0.95
NUM_TRAINING_EPOCHS = 30

# Loss weights
W_3D = 1000.0     # 3D joint loss
W_2D = 10.0       # 2D reprojection loss
W_PARAM = 1.0     # Parameter regularization

# Checkpoint
SAVE_BEST_ONLY = True
SAVE_PERIODIC = True
PERIODIC_SAVE_INTERVAL = 5

print("Configuration loaded:")
print(f"  BATCH_SIZE={BATCH_SIZE}")
print(f"  BACKBONE_MODEL={BACKBONE_MODEL}")
print(f"  NUM_TRAINING_EPOCHS={NUM_TRAINING_EPOCHS}")
print(f"  W_3D={W_3D}, W_2D={W_2D}, W_PARAM={W_PARAM}")

## 4. Build Dataset

In [ ]:
from data.dataset import build_dataset

print("Building training dataset...")
print(f"  Batch size: {BATCH_SIZE}")

train_dataset = build_dataset(
    batch_size=BATCH_SIZE,
    shuffle=True,
    training=True,
)

# Prefetch for performance
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# Check sample
sample_batch = next(iter(train_dataset.take(1)))
print("\nDataset sample:")
print(f"  image: {sample_batch['image'].shape}")
print(f"  keypoints: {sample_batch['keypoints'].shape}")
print(f"  K: {sample_batch['K'].shape}")

## 5. Build Model and Loss

In [ ]:
# Create model
print("Creating Shape Network...")
model = create_shape_net(
    backbone_name=BACKBONE_MODEL,
    trainable_backbone=TRAINABLE_BACKBONE,
)

# Optimizer
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=INITIAL_LEARNING_RATE,
    decay_steps=DECAY_STEPS,
    decay_rate=DECAY_RATE,
    staircase=True,
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# Loss function
from training.losses import shape_loss

print("Model and optimizer ready")
print(f"Total trainable params: {model.count_params():,}")

## 6. Training Loop

In [ ]:
from datetime import datetime
from pathlib import Path

# Directories
ckpt_dir = REPO_ROOT / "checkpoints" / "shape_net"
ckpt_dir.mkdir(parents=True, exist_ok=True)

log_dir = REPO_ROOT / "logs" / "shape_net" / datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir.mkdir(parents=True, exist_ok=True)

summary_writer = tf.summary.create_file_writer(str(log_dir))

best_loss = float("inf")
global_step = 0

print("=" * 60)
print("TRAINING START")
print("=" * 60)
print(f"Epochs: {NUM_TRAINING_EPOCHS}")
print(f"Checkpoint dir: {ckpt_dir}")
print(f"Log dir: {log_dir}")

try:
    for epoch in range(NUM_TRAINING_EPOCHS):
        epoch_losses = []
        
        for batch_idx, batch in enumerate(train_dataset):
            images = batch["image"]
            gt_keypoints = batch["keypoints"]  # (B, 21, 3) xyz
            K = batch["K"]  # (B, 3, 3)
            
            with tf.GradientTape() as tape:
                # Forward pass
                pred_params = model(images, training=True)
                
                # Compute loss
                loss = shape_loss(
                    pred_params,
                    gt_keypoints,
                    K=K,
                    w_3d=W_3D,
                    w_2d=W_2D,
                    w_p=W_PARAM,
                    use_mano=True,
                )
            
            # Backward pass
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            
            loss_value = float(loss.numpy())
            epoch_losses.append(loss_value)
            global_step += 1
            
            if (batch_idx + 1) % 10 == 0:
                with summary_writer.as_default():
                    tf.summary.scalar("loss/train_step", loss_value, step=global_step)
        
        # Epoch summary
        avg_loss = float(np.mean(epoch_losses))
        current_lr = float(lr_schedule(optimizer.iterations).numpy())
        
        with summary_writer.as_default():
            tf.summary.scalar("loss/train_epoch", avg_loss, step=epoch)
            tf.summary.scalar("learning_rate", current_lr, step=epoch)
        
        print(f"Epoch {epoch+1:02d}/{NUM_TRAINING_EPOCHS} | loss={avg_loss:.6f} | lr={current_lr:.3e}")
        
        # Save checkpoint
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_path = ckpt_dir / "best.weights.h5"
            model.save_weights(str(best_path))
            print(f"  → Saved best: {best_path.name} (loss={best_loss:.6f})")
        
        if SAVE_PERIODIC and (epoch + 1) % PERIODIC_SAVE_INTERVAL == 0:
            periodic_path = ckpt_dir / f"epoch_{epoch+1:03d}.weights.h5"
            model.save_weights(str(periodic_path))
            print(f"  → Saved periodic: {periodic_path.name}")

except KeyboardInterrupt:
    print("\nTraining interrupted by user")
except Exception as e:
    import traceback
    print(f"\nTraining failed: {e}")
    traceback.print_exc()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Best loss: {best_loss:.6f}")
print(f"Checkpoint: {ckpt_dir / 'best.weights.h5'}")

## 7. Summary

In [ ]:
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)

# List checkpoints
if ckpt_dir.exists():
    checkpoints = sorted(ckpt_dir.glob("*.h5"))
    print(f"\nCheckpoints saved in: {ckpt_dir}")
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / (1024**2)
        print(f"  - {ckpt.name:30s} {size_mb:.1f} MB")

print(f"\nTensorBoard log dir: {log_dir}")
print("\nTo view training curves:")
print("  tensorboard --logdir logs/shape_net --port 6006")
print("=" * 60)